:::{admonition} Download
:class: important

Download this notebook: **{nb-download}`single_neuron.ipynb`**!

:::

In [1]:
%matplotlib inline
import warnings

# Ignore the first specific warning
warnings.filterwarnings(
    "ignore",
    message="plotting functions contained within `_documentation_utils` are intended for nemos's documentation.",
    category=UserWarning,
)

# Ignore the second specific warning
warnings.filterwarnings(
    "ignore",
    message="Ignoring cached namespace 'core'",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=(
        "invalid value encountered in div "
    ),
    category=RuntimeWarning,
)

(glm-single-neuron)=
# Generalized Linear Model (GLM)

This page gives an overview of the GLM features in NeMoS.

```{image} ../../../assets/glm.svg
:alt: GLM. X of shape (n_samples, n_features) times a weight vector w, through the inverse link f, gives one output y of shape (n_samples,).
:class: only-light
```

```{image} ../../../assets/glm_dark.svg
:alt: GLM. X of shape (n_samples, n_features) times a weight vector w, through the inverse link f, gives one output y of shape (n_samples,).
:class: only-dark
```

```{contents}
:local:
:depth: 3
```

## Initializing a GLM object

Every argument of [`GLM`](nemos.glm.GLM) has a default value. Without arguments, the model is a Poisson GLM with no regularization.

In [2]:
import nemos as nmo

nmo.glm.GLM()

,observation_model,PoissonObservations()
,inverse_link_function,<function exp...x7fd2965c22a0>
,regularizer,UnRegularized()
,fix_params,"(None, ...)"
,solver_name,'LBFGS'
,solver_kwargs,{}
,regularizer_strength,None
,fit_intercept,True
Name,Type,Value
aux_,NoneType,None
coef_,NoneType,None


The arguments describe two things:

- The data: the observation model and the inverse link function.
- The estimation of the parameters: the regularizer and the solver.

[Model configurations](glm-model-configuration) explains each argument.

In [3]:
model = nmo.glm.GLM(
    observation_model="Gamma",
    regularizer="Ridge",
    regularizer_strength=0.1,
    solver_name="Newton",
)
model

,observation_model,GammaObservations()
,inverse_link_function,<function one...x7fd2965c14e0>
,regularizer,Ridge()
,regularizer_strength,0.1
,fix_params,"(None, ...)"
,solver_name,'Newton'
,solver_kwargs,{}
,fit_intercept,True
Name,Type,Value
aux_,NoneType,None
coef_,NoneType,None


(glm-input-types)=
## Inputs

`y` is always an array. `X` can be an array, or a container of arrays with one group of predictors per leaf. JAX calls this container a [PyTree](https://docs.jax.dev/en/latest/pytrees.html) of arrays. The supported array formats are [numpy](https://numpy.org/), [jax](https://docs.jax.dev/), and [pynapple](https://www.pynapple.org) time series.

### Arrays

`X` has shape `(n_samples, n_features)` and `y` has shape `(n_samples,)`. Each of them can be a NumPy array, a JAX array, or a [pynapple](https://www.pynapple.org) time series.

#### NumPy

In [4]:
import numpy as np

# X of shape (n_samples, n_features), y of shape (n_samples,)
X, y = nmo.datasets.tuned_poisson(n_samples=500)  # JAX arrays
X_np, y_np = np.asarray(X), np.asarray(y)

model_np = nmo.glm.GLM().fit(X_np, y_np)

The fitted parameters and the output of `predict` are JAX arrays, also for a NumPy input.

In [5]:
type(model_np.predict(X_np))

jaxlib._jax.ArrayImpl

#### JAX

`fit` gives the same parameters for JAX arrays and for NumPy arrays.

In [6]:
model_jax = nmo.glm.GLM().fit(X, y)

np.allclose(model_jax.coef_, model_np.coef_)

True

#### Pynapple

`fit` also accepts [pynapple](https://www.pynapple.org) time series. A [pynapple](https://www.pynapple.org) object stores the timestamps with the values. It also stores a time support, which records the epochs of the session that the samples come from.

In [7]:
import pynapple as nap

# suppress jax to numpy conversion warning
nap.nap_config.suppress_conversion_warnings = True

# 10 ms bins
time = np.arange(X.shape[0]) / 100

# two epochs of different duration, labeled by trial type
epochs = nap.IntervalSet(
    start=[0.0, 3.0],
    end=[1.5, 5.],
    metadata={"contrast": ["high", "low"]},
)

predictors = nap.TsdFrame(t=time, d=np.asarray(X), time_support=epochs)
counts = nap.Tsd(t=time, d=np.asarray(y), time_support=epochs)

model_nap = nmo.glm.GLM().fit(predictors, counts)
model_nap

,observation_model,PoissonObservations()
,inverse_link_function,<function exp...x7fd2965c22a0>
,regularizer,UnRegularized()
,fix_params,"(None, ...)"
,solver_name,'LBFGS'
,solver_kwargs,{}
,regularizer_strength,None
,fit_intercept,True
Name,Type,Value
aux_,NoneType,None
coef_,"ArrayImpl[float32](8,)",Array([-1.983...dtype=float32)


If the input is a [pynapple](https://www.pynapple.org) object, [`predict`](nemos.glm.GLM.predict) and [`simulate`](nemos.glm.GLM.simulate) return a [pynapple](https://www.pynapple.org) object.

### PyTrees

`X` can also be a PyTree of arrays, with one leaf per group of predictors. All leaves must have the same number of samples. You do not need to concatenate the groups into a single design matrix. The coefficients have the same structure as `X`. Examples of PyTrees are dictionaries, lists, and your own classes registered as PyTree nodes.

With a dictionary:

In [8]:
X_dict = {"stimulus": X[:, :4], "history": X[:, 4:]}

model_dict = nmo.glm.GLM().fit(X_dict, y)

model_dict.coef_

{'history': Array([ 1.1424748 ,  0.36609316, -0.5771389 , -0.98125434], dtype=float32),
 'stimulus': Array([-1.7550181 , -0.2239651 ,  0.13521995,  1.3519512 ], dtype=float32)}

JAX flattens a dictionary in sorted key order. Thus, `history` comes before `stimulus`, whatever the order of the keys in the dictionary.

With a list:

In [9]:
# swap array order to match dict print order
model_list = nmo.glm.GLM().fit([X[:, 4:], X[:, :4]], y)

model_list.coef_

[Array([ 1.1424748 ,  0.36609316, -0.5771389 , -0.98125434], dtype=float32),
 Array([-1.7550181 , -0.2239651 ,  0.13521995,  1.3519512 ], dtype=float32)]

With your own class, registered as a PyTree node:

In [10]:
import equinox as eqx
from jax import Array


class Input(eqx.Module):
    stimulus: Array
    history: Array


model_eqx = nmo.glm.GLM().fit(Input(X[:, :4], X[:, 4:]), y)

model_eqx.coef_

Input(stimulus=f32[4], history=f32[4])

The leaves can be [pynapple](https://www.pynapple.org) objects. In this case, the predicted rate keeps the time axis.

In [11]:
X_nap_tree = [predictors[:, :4], predictors[:, 4:]]

model_nap_tree = nmo.glm.GLM().fit(X_nap_tree, counts)

model_nap_tree.predict(X_nap_tree)

Time (s)
----------  --------
0.0         0.517009
0.01        0.717208
0.02        8.90007
0.03        8.60559
0.04        4.45489
0.05        2.72407
0.06        2.83755
...
4.93        4.00572
4.94        4.44582
4.95        3.30842
4.96        9.82204
4.97        2.00648
4.98        3.1202
4.99        9.4668
dtype: float32, shape: (351,)

(glm-methods)=
## Methods

### Fit

[`fit`](nemos.glm.GLM.fit) estimates the parameters and returns the model. You can call it directly on the constructor.

In [12]:
model = nmo.glm.GLM().fit(X, y)
model

,observation_model,PoissonObservations()
,inverse_link_function,<function exp...x7fd2965c22a0>
,regularizer,UnRegularized()
,fix_params,"(None, ...)"
,solver_name,'LBFGS'
,solver_kwargs,{}
,regularizer_strength,None
,fit_intercept,True
Name,Type,Value
aux_,NoneType,None
coef_,"ArrayImpl[float32](8,)",Array([-1.755...dtype=float32)


#### Fitted attributes

The fit stores the estimated quantities as attributes with a trailing underscore.

In [13]:
print(f"coef_:       {model.coef_.shape}")
print(f"intercept_:  {model.intercept_.shape}")
print(f"scale_:      {model.scale_}")
print(f"dof_resid_:  {model.dof_resid_}")

coef_:       (8,)
intercept_:  (1,)
scale_:      [1.]
dof_resid_:  [491.]


- `coef_`: one weight per column of the design matrix, on the scale of the linear term.
- `intercept_`: the baseline, on the scale of the linear term.
- `scale_`: the estimated dispersion.
- `dof_resid_`: the residual degrees of freedom.
- `solver_state_`: the state of the solver at the end of the fit.

#### NaN treatment

The fit ignores samples with a NaN in `X` or `y`. Examples of such samples are bins without a measurement, and the edges that a convolution leaves undefined. You do not need to remove these samples by hand.

In [14]:
X_nan = np.asarray(X).copy()
X_nan[:10] = np.nan

model_nan = nmo.glm.GLM().fit(X_nan, y)
model_dropped = nmo.glm.GLM().fit(np.asarray(X)[10:], np.asarray(y)[10:])

np.allclose(model_nan.coef_, model_dropped.coef_)

True

### Predict

[`predict`](nemos.glm.GLM.predict) returns the firing rate in counts per bin, one value per sample.

In [15]:
rate = model.predict(X)
rate.shape

(500,)

To convert the rate to spikes per second, divide it by the bin size.

In [16]:
bin_size = 0.01  # 10 ms bins, in seconds

rate[:10] / bin_size  # rate in Hz

Array([ 63.48378,  83.58063, 929.66473, 844.0752 , 444.47626, 269.49762,
       280.29132, 852.4475 , 224.21143, 129.55971], dtype=float32)

The output type matches the type of the predictors given to `predict`. The type of the data used for the fit does not matter.

In [17]:
print(type(model_nap.predict(X)))
print(type(model_nap.predict(predictors)))

<class 'jaxlib._jax.ArrayImpl'>
<class 'pynapple.core.time_series.Tsd'>


A [pynapple](https://www.pynapple.org) output keeps the time axis of the predictors. You can restrict it to an epoch, or cross-correlate it with other time series, without adding the timestamps again. The timestamps come from the predictors given to `predict`. These predictors can cover any interval.

### Score

[`score`](nemos.glm.GLM.score) evaluates the model on data. By default, it returns the mean log-likelihood per sample.

In [18]:
model.score(X, y)

Array(-2.0672421, dtype=float32)

Set `score_type` to use one of the two pseudo-$R^2$ scores instead. Unlike the log-likelihood, a pseudo-$R^2$ has a maximum of 1, and you can compare it across datasets.

In [19]:
model.score(X, y, score_type="pseudo-r2-McFadden")

Array(0.32585198, dtype=float32)

`aggregate_sample_scores` combines the per-sample scores into one value. The default is the mean. The function receives the array of per-sample scores. Thus, the identity function returns this array instead of a number.

In [20]:
model.score(X, y, aggregate_sample_scores=lambda x: x).shape

(500,)

The sum gives the total log-likelihood over the samples.

In [21]:
import jax.numpy as jnp

model.score(X, y, aggregate_sample_scores=jnp.sum)

Array(-1033.6211, dtype=float32)

### Simulate

[`simulate`](nemos.glm.GLM.simulate) draws counts from the observation model at the predicted rate. It returns the counts and the rate.

In [22]:
import jax

counts_sim, rate_sim = model.simulate(jax.random.key(0), X)

print(f"counts: {counts_sim[:5]}")
print(f"rate:   {np.round(np.asarray(rate_sim[:5]), 2)}")

counts: [ 0  0 10  8  2]
rate:   [0.63 0.84 9.3  8.44 4.44]


The random key makes the draw reproducible: the same key and the same rate give the same counts.

:::{admonition} Simulating a coupled population
:class: dropdown

[`simulate`](nemos.glm.GLM.simulate) is feed-forward: the simulated counts do not change the rate. To simulate a population in which spikes drive the next time step through coupling filters, use [`simulate_recurrent`](nemos.simulation.simulate_recurrent). See [simulate a coupled population](../../../how_to_guide/simulate_coupled_population.md).
:::

(glm-out-of-memory)=
### Fitting out of memory

If the design matrix does not fit in memory, use [`stochastic_fit`](nemos.glm.GLM.stochastic_fit). It takes a data loader instead of arrays, and updates the parameters one batch at a time. Stochastic solvers require acceleration and line searches to be off.

:::{admonition} In-memory example
:class: note

The arrays here fit in memory, so the loader is an [`ArrayDataLoader`](nemos.batching.ArrayDataLoader). For data that does not fit in memory, [`LazyArrayDataLoader`](nemos.batching.LazyArrayDataLoader) loads one batch at a time from a dask, zarr or HDF5 array.
:::

In [23]:
# the loader serves the data in batches of 100 samples
loader = nmo.batching.ArrayDataLoader(X, y, batch_size=100)

glm = nmo.glm.GLM(
    solver_name="GradientDescent",
    solver_kwargs={"stepsize": 0.05, "acceleration": False},
)
glm.stochastic_fit(loader, n_passes=20)

,observation_model,PoissonObservations()
,inverse_link_function,<function exp...x7fd2965c22a0>
,regularizer,UnRegularized()
,fix_params,"(None, ...)"
,solver_name,'GradientDescent'
,solver_kwargs,"{'acceleration': False, 'stepsize': 0.05}"
,regularizer_strength,None
,fit_intercept,True
Name,Type,Value
aux_,NoneType,None
coef_,"ArrayImpl[float32](8,)",Array([-0.670...dtype=float32)


See [stochastic optimization](../../scalability/README.md).

#### Update

For full control of the optimization loop, call [`update`](nemos.glm.GLM.update) to do each step yourself. [`initialize_params`](nemos.glm.GLM.initialize_params) returns the initial parameters. [`initialize_optimizer_and_state`](nemos.glm.GLM.initialize_optimizer_and_state) sets up the solver.

In [24]:
params = glm.initialize_params(X, y)
opt_state = glm.initialize_optimizer_and_state(params, X, y)

# a full pass over all the batches
for X_batch, y_batch in loader:
    params, opt_state = glm.update(params, opt_state, X_batch, y_batch)

The loop body can contain any code. Examples are a log of the score for each batch, a change of the learning rate between passes, or your own stopping rule. See [custom update](../../scalability/custom_update.md).

(glm-model-configuration)=
## Model configurations

(glm-observation-model)=
### Observation models

The observation model is the distribution of the counts around the predicted rate. You can set it by name:

In [25]:
nmo.glm.GLM("Gamma")

,observation_model,GammaObservations()
,inverse_link_function,<function one...x7fd2965c14e0>
,regularizer,UnRegularized()
,fix_params,"(None, ...)"
,solver_name,'LBFGS'
,solver_kwargs,{}
,regularizer_strength,None
,fit_intercept,True
Name,Type,Value
aux_,NoneType,None
coef_,NoneType,None


You can also give the observation model object. The object gives access to the parameters of the family. The Negative Binomial needs the object, because its `dispersion` is a fixed value, not an estimated one.

In [26]:
from nemos.observation_models import NegativeBinomialObservations

nmo.glm.GLM(NegativeBinomialObservations(dispersion=2.0))

,observation_model,NegativeBinom...ispersion=2.0)
,inverse_link_function,<function exp...x7fd2965c22a0>
,regularizer,UnRegularized()
,fix_params,"(None, ...)"
,solver_name,'LBFGS'
,solver_kwargs,{}
,observation_model__dispersion,2.0
,regularizer_strength,None
,fit_intercept,True
Name,Type,Value
aux_,NoneType,None


[Observation models](../../observation_models.md) describes each family, the signals it suits, and what it estimates.

(glm-inverse-link)=
### Inverse link function

The inverse link function maps the linear term to the mean of the observation model. Each family has a default. Only a few other choices keep the log-likelihood convex.

| Observation model | Default | Convexity preserving |
| --- | --- | --- |
| `Poisson` | `jnp.exp` | `jnp.exp`, `jax.nn.softplus` |
| `Gamma` | `one_over_x` | `jnp.exp`, `one_over_x` |
| `Gaussian` | `identity` | `identity` |
| `Bernoulli` | `logistic` | `logistic`, `expit` |
| `NegativeBinomial` | `jnp.exp` | `jnp.exp` |

:::{admonition} Convexity depends on the pair
:class: warning

`jax.nn.softplus` is safe for a Poisson GLM, but it makes the Negative Binomial log-likelihood non-convex. NeMoS does not reject such a model. The fit runs, but it converges to a local optimum that depends on the starting point. Also, the solver can no longer rely on a positive semi-definite curvature. NeMoS treats a link that is not in the table the same way.
:::

In [27]:
# convex
nmo.glm.GLM("Poisson", inverse_link_function=jax.nn.softplus)

# not convex
nmo.glm.GLM("NegativeBinomial", inverse_link_function=jax.nn.softplus)

,observation_model,NegativeBinom...ispersion=1.0)
,inverse_link_function,<PjitFunction...7fd2bc44b7e0>>
,regularizer,UnRegularized()
,fix_params,"(None, ...)"
,solver_name,'LBFGS'
,solver_kwargs,{}
,observation_model__dispersion,1.0
,regularizer_strength,None
,fit_intercept,True
Name,Type,Value
aux_,NoneType,None


(glm-regularizers)=
### Regularizers

Set the regularizer and its strength when you create the model. If the regularizer needs a strength and `regularizer_strength` is not set, NeMoS gives a warning and uses 1.

In [28]:
nmo.glm.GLM(regularizer="Lasso", regularizer_strength=0.05).fit(X, y).coef_

Array([-0.8237829 , -0.        ,  0.        ,  1.3647242 ,  1.2954941 ,
        0.        , -0.        , -0.13412239], dtype=float32)

[Regularizers](../../regularizers.md) describes group-wise and parameter-wise strengths, and the special cases of ElasticNet and GroupLasso.

(glm-solvers)=
### Solvers

Select the solver by name, and configure it with `solver_kwargs`.

In [29]:
model = nmo.glm.GLM(solver_name="Newton", solver_kwargs={"tol": 1e-10}).fit(X, y)

The keys of `solver_kwargs` must be arguments of the solver. Other keys raise an error. To see the accepted arguments, print the solver docstring with [`get_solver_documentation`](nemos.solvers.get_solver_documentation). The next cell prints the start of the `LBFGS` docstring, up to the list of accepted arguments.

In [30]:
from nemos.solvers import get_solver_documentation

doc = get_solver_documentation("LBFGS")
print("\n".join(doc.splitlines()[:27]))

Showing docstring of nemos.solvers._optax_optimistix_solvers.OptimistixOptaxLBFGS.
For potentially more info, use `show_help=True`.

L-BFGS implementation using optax.lbfgs wrapped by optimistix.OptaxMinimiser.

Convergence criterion is implemented by Optimistix, so the Cauchy criterion is used.

Accepted arguments:
-------------------
- adjoint
- has_aux
- init_params
- maxiter
- memory_size
- norm
- optim
- options
- rtol
- scale_init_precond
- stepsize
- tags
- throw
- tol
- verbose

Note that for backward compatibility the `atol` parameter used in Optimistix
is referred to as `tol` in NeMoS.


If you do not set the solver, NeMoS selects one that is compatible with the regularizer:

- `Newton` for Ridge.
- `LBFGS` without regularization.
- `ProximalGradient` for the non-smooth penalties.

In [31]:
print(nmo.glm.GLM().solver_name)
print(nmo.glm.GLM(regularizer="Ridge", regularizer_strength=0.01).solver_name)
print(nmo.glm.GLM(regularizer="Lasso", regularizer_strength=0.01).solver_name)

LBFGS
Newton
ProximalGradient


The registry lists the installed solvers as `algo_name` and `backend` pairs.

In [32]:
from nemos.solvers import list_available_solvers

sorted(f"{spec.algo_name}[{spec.backend}]" for spec in list_available_solvers())

['BFGS[optimistix]',
 'GradientDescent[optax+optimistix]',
 'GradientDescent[optimistix]',
 'LBFGS[optax+optimistix]',
 'Newton[nemos]',
 'NonlinearCG[optimistix]',
 'ProxSVRG[nemos]',
 'ProximalGradient[optimistix]',
 'ProximalNewton[nemos]',
 'SVRG[nemos]']

[Solvers](../../solvers.md) describes which solver accepts which regularizer, which solver to use, and how to register your own solver.

(glm-parameter-specification)=
## Parameter specification

### Drop the intercept

`fit_intercept=False` removes the intercept from the model. The intercept stays at zero and uses no degree of freedom.

In [33]:
no_intercept = nmo.glm.GLM(fit_intercept=False).fit(X, y)

print(f"intercept_: {no_intercept.intercept_}")
print(f"dof_resid_: {no_intercept.dof_resid_}")

intercept_: [0.]
dof_resid_: [492.]


### Fix parameter values

`fix_params` keeps any leaf of the parameters at a value that you choose. The parameters are a `(coef, intercept)` tuple. If `X` is a PyTree, `coef` is also a PyTree. Thus, you can fix these leaves:

- the intercept,
- each leaf of `coef`.

For a leaf to fix, give its array of values. For a leaf to learn, give `None` as a placeholder.

If `X` is an array, `coef` is a single leaf. The fit fixes or learns all of it.

#### Fix a value for the intercept

In [34]:
# hold the intercept at 0.5, learn the weights
model_fixed = nmo.glm.GLM(fix_params=(None, np.array([0.5]))).fit(X, y)

print(f"coef_:      {model_fixed.coef_}")
print(f"intercept_: {model_fixed.intercept_}")

coef_:      [-1.1166209   0.4145253   0.773631    1.9904119   1.7809154   1.0045357
  0.06131251 -0.34280768]
intercept_: [0.5]


#### Fix a value for the coefficients

In [35]:
# hold all the weights at zero, learn the intercept
model_fixed = nmo.glm.GLM(fix_params=(np.zeros(8), None)).fit(X, y)

print(f"coef_:      {model_fixed.coef_}")
print(f"intercept_: {model_fixed.intercept_}")

coef_:      [0. 0. 0. 0. 0. 0. 0. 0.]
intercept_: [1.6800827]


#### Partially specify the coefficients

If `X` is a PyTree, you can fix or learn each leaf of `coef` separately.

In [36]:
# hold the history weights at zero, learn the stimulus ones
fixed = ({"stimulus": None, "history": np.zeros(4)}, None)

model_fixed = nmo.glm.GLM(fix_params=fixed).fit(X_dict, y)

print(f"coef_:      {model_fixed.coef_}")
print(f"intercept_: {model_fixed.intercept_}")

coef_:      {'history': Array([0., 0., 0., 0.], dtype=float32), 'stimulus': Array([-2.4657936 , -0.15728971, -0.8669459 ,  1.5429988 ], dtype=float32)}
intercept_: [1.5372783]


:::{admonition} Example use case
:class: dropdown

Fit the same design two times: one time with a group of predictors fixed at zero, and one time with this group free. The result is two nested models. Compare them to test whether that group explains the activity.
:::

#### Interaction between `fit_intercept` and `fix_params`

An intercept set with `fix_params` has priority over `fit_intercept`. With both arguments, the intercept stays at the value in `fix_params`, not at zero.

In [37]:
both = nmo.glm.GLM(fit_intercept=False, fix_params=(None, np.array([0.5]))).fit(X, y)
both.intercept_

Array([0.5], dtype=float32)

## Examples

```{nemos-examples}
:model: GLM
```